Point 3: NER & Keyphrase Extraction

Read the text from the PDF and automatically find:
- Important named things (like Vitamin C, Fish, Red meat), done with NER (Named Entity Recognition);
- Important phrases or concepts that might not match exactly the KB (like iodised salt, wholegrain rye bread), done with keyphrase extraction.

NER: using spaCy library (can mark words or phrases as certain types). Using an EntityRuler that contains patterns like "Vitamin C", "red meat", then running the model over the text chunks extracted from the PDF and contained in the raw_text.json. spaCy will label words it recognizes with custom categories.

Keyphrase extraction: use libraries like YAKE (finds keywords just by analyzing word frequency and patterns) and KeyBERT (finds key phrases that are semantically important using a transformer), feed each text chunk to YAKE or KeyBERT and it will give you the most important phrases. Then compare these phrases with your KB aliases, if a phrase isn’t already in your KB, note it down — it could become a new alias or even a new entity later.

Cell 1 — Imports, configuration, and helper functions
This cell loads your Python libraries, defines file paths, and adds small helper functions for normalization and validation.
You only need to run this once at the top of your notebook.

In [1]:
!pip install yake keybert

In [ ]:
# --- Point 3 · Step 1: setup environment & helpers -------------------------

from __future__ import annotations
import json, re
from pathlib import Path
from typing import Dict, List, Iterable, Any
from collections import defaultdict
import yake
from keybert import KeyBERT

# --- Configuration: adjust if your files are in another folder -------------
KB_PATH = Path("data/kb_extended.json")            # knowledge base (canonical entities + aliases)
RAW_TEXT_PATH = Path("data/raw_text.jsonl") # text chunks extracted from PDF
ONTOLOGY_PATH = Path("data/ontologyv2.yaml")# ontology schema
DEFAULT_DOC_ID = "SUSTAINABLE_HEALTH_FROM_FO    OD"


def norm(text: str) -> str:
    """Normalize surface forms for alias lookup (lowercase, collapse whitespace & punctuation)."""
    t = text.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[‐–—−]", "-", t)
    t = re.sub(r"[“”]", '"', t).replace("’","'")
    return t.strip()

# ensures kb entry has the correct structure
def validate_kb_entry(e: Dict[str, Any]) -> List[str]:
    """Light validation for one KB entry."""
    errs = []
    if not {"id","label","class"} <= e.keys():
        errs.append("Missing required keys (id/label/class).")
    else:
        if not re.match(r"^ex:[a-zA-Z][a-zA-Z0-9_]*\.[a-z0-9_]+$", e["id"]):
            errs.append(f"Bad id format: {e['id']}")
    if "aliases" in e and not isinstance(e["aliases"], list):
        errs.append("aliases must be a list.")
    return errs


/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cell 2 — Load and validate your KB
This cell reads kb-2.json, removes duplicate aliases, validates IDs, and reports how many entities you have.
It also builds two key structures you’ll use later:
entityruler_patterns for spaCy’s EntityRuler
alias_index for linking spans to canonical IDs

In [ ]:
# --- Load KB & basic validation --------------------------------------------

def load_kb(kb_path: Path) -> List[Dict[str, Any]]:
    data = json.loads(kb_path.read_text(encoding="utf-8"))
    for e in data:
        seen, dedup = set(), []
        for a in e.get("aliases", []):
            if a and a not in seen:
                seen.add(a); dedup.append(a)
        e["aliases"] = dedup
    return data


# converts KB entries into patterns for spaCy’s EntityRuler:
# each pattern includes:
# label → the entity type (e.g., NUTRIENT)
# pattern → text string to match
def build_entityruler_patterns(kb: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Return patterns ready to add to spaCy EntityRuler."""
    pats = []
    for ent in kb:
        label = ent["class"].upper()
        for s in [ent["label"], *ent["aliases"]]:
            if s and s.strip():
                pats.append({"label": label, "pattern": s})
    return pats


# Creates a lookup dictionary:
# Key → normalized string (norm(s)).
# Value → list of KB entries that match that string.
def build_alias_index(kb: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, str]]]:
    """normalized surface -> list of {id,class,label} dicts."""
    idx = defaultdict(list)
    for ent in kb:
        for s in [ent["label"], *ent["aliases"]]:
            key = norm(s)
            if key:
                idx[key].append({"id": ent["id"], "class": ent["class"], "label": ent["label"]})
    return idx

# --- Execute load ----------------------------------------------------------
kb = load_kb(KB_PATH)
entityruler_patterns = build_entityruler_patterns(kb)
alias_index = build_alias_index(kb)

print(f"KB entries loaded: {len(kb)}")
print(f"EntityRuler patterns: {len(entityruler_patterns)}")
print(f"Alias index entries: {len(alias_index)}")

# Quick check for malformed entries
errors = []
for e in kb:
    errs = validate_kb_entry(e)
    if errs:
        errors.append((e["id"], errs))
if errors:
    print(f"[WARN] {len(errors)} KB entries with issues (showing 3):", errors[:3])
else:
    print("KB validation: OK")


KB entries loaded: 110
EntityRuler patterns: 396
Alias index entries: 280
KB validation: OK


Cell 3 — Define label↔class map, over-broad terms, and context cues
This small config keeps your ontology classes aligned with NER labels,
and prepares guardrails for generic terms like “sugar” or “fish”.

In [ ]:
# --- Mappings and guardrails ----------------------------------------------

LABEL_TO_CLASS = {
    "NUTRIENT": "nutrient",
    "FOOD_GROUP": "foodGroup",
    "FOOD_ITEM": "foodItem",
    "TECHNIQUE": "technique",
}
CLASS_TO_LABEL = {v:k for k,v in LABEL_TO_CLASS.items()}

# Over-broad terms that might match too often (you’ll filter later)
OVERBROAD = {"sugar","protein","fish","meat","fat","fats","oil","oils",
             "juice","juices","drink","drinks","salt"}

# Context cues for validating those generic matches in Step 4
CONTEXT_CUES = {
    "nutrient": {"intake","rda","ai","ul","e%","deficiency","vitamin","mineral"},
    "foodGroup": {"portion","g/day","g/week","eat","limit","increase","reduce","serving"},
    "foodItem": {"portion","eat","limit","increase","reduce","serving"},
    "technique": {"fried","boiled","baked","cooked","prepared","processing"},
}

print("Mappings and guardrails ready.")


Mappings and guardrails ready.


Cell 4 — Prepare iterator over raw_text.jsonl
This cell streams your extracted text chunks one by one so you don’t load the whole file into memory.
It also fills missing docId fields with a default value.

In [ ]:
# --- Iterator for raw_text.jsonl ------------------------------------------

def iter_raw_text(path: Path, default_doc_id: str):
    """Yield dicts with docId, page, section, text (skip invalid lines)."""
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = obj.get("text")
            page = obj.get("page")
            section = obj.get("section", "")
            if text is None or page is None:
                continue
            doc_id = obj.get("docId", default_doc_id)
            yield {
                "docId": doc_id,
                "page": page,
                "section": section,
                "text": text,
            }

# Build the iterator and preview a couple of chunks
raw_stream = iter_raw_text(RAW_TEXT_PATH, DEFAULT_DOC_ID)

first_two = []
for _ in range(2):
    try:
        first_two.append(next(raw_stream))
    except StopIteration:
        break

print("Sample text chunks:")
for i, rec in enumerate(first_two, 1):
    print(f"[{i}] page={rec['page']} section='{rec['section']}'")
    print("   text:", rec['text'][:120].replace("\n"," "), "...\n")

# Re-chain the first two back into the iterator for later steps
def chain_iters(first, rest_iter):
    for x in first:
        yield x
    for x in rest_iter:
        yield x

raw_stream = chain_iters(first_two, iter_raw_text(RAW_TEXT_PATH, DEFAULT_DOC_ID))


Sample text chunks:
[1] page=5 section='Preface'
   text: The aim of the nutrition recommendations is to promote public health. The recommendations are widely used in healthcare, ...

[2] page=6 section='Preface'
   text: High-quality monitoring of food consumption has a long tradition in Finland. In the recommendations, the data on food co ...



Cell 5 — Initialize output buffers and summary
This final setup cell creates empty lists for linked/unlinked mentions and prints a quick summary.
These structures will be filled in Steps 3–6 of your pipeline.

In [ ]:
# --- Output scaffolding and summary ---------------------------------------

mentions_buffer: List[Dict[str, Any]] = []   # linked mentions (filled later)
debug_unlinked: List[Dict[str, Any]] = []    # unlinked mentions for KB growth

kb_classes = sorted({e["class"] for e in kb})

print("=== Step 1: setup complete ===")
print(f"KB classes present: {', '.join(kb_classes)}")
print(f"EntityRuler patterns ready: {len(entityruler_patterns)}")
print(f"Alias index entries: {len(alias_index)}")
print("Label↔class mappings:", LABEL_TO_CLASS)
print("Context cues prepared for:", ", ".join(CONTEXT_CUES.keys()))
print("Iterator ready for raw_text.jsonl.")
print("mentions_buffer and debug_unlinked initialized (empty).")


=== Step 1: setup complete ===
KB classes present: dietaryGuideline, environmentImpact, foodGroup, foodItem, nutrient, populationGroup, technique
EntityRuler patterns ready: 396
Alias index entries: 280
Label↔class mappings: {'NUTRIENT': 'nutrient', 'FOOD_GROUP': 'foodGroup', 'FOOD_ITEM': 'foodItem', 'TECHNIQUE': 'technique'}
Context cues prepared for: nutrient, foodGroup, foodItem, technique
Iterator ready for raw_text.jsonl.
mentions_buffer and debug_unlinked initialized (empty).


In [ ]:
# --- Entity linking via alias index ---------------------------------------
# For each text chunk, find all substrings that match any alias in the alias index.

def find_entities_in_text(text: str, alias_index: dict):
    """
    Find all KB entities mentioned in a text chunk, using n-grams up to 4 words.
    Returns matches with character positions.
    """
    matches = []
    words = re.split(r'(\W+)', text)  # split but keep delimiters for accurate positions

    # Build cumulative character positions
    char_positions = []
    pos = 0
    for w in words:
        char_positions.append(pos)
        pos += len(w)

    # Iterate over word spans (n-grams)
    for i in range(0, len(words), 2):  # skip delimiter positions
        for j in range(i+1, min(i+8, len(words)+1), 2):  # up to 4 words
            span_words = words[i:j:2]  # only actual words
            span = "".join(span_words)
            key = norm(span)
            if key in alias_index:
                start_char = char_positions[i]
                end_char = char_positions[j-1] + len(words[j-1])
                for ent in alias_index[key]:
                    matches.append({
                        "text": span,
                        "id": ent["id"],
                        "class": ent["class"],
                        "label": ent["label"],
                        "start_char": start_char,
                        "end_char": end_char
                    })
    return matches



In [ ]:
import pandas as pd

# Only keep generic words if context words exist nearby in the text.

def filter_overbroad_matches(matches, text):
    filtered = []
    for m in matches:
        if m['text'].lower() in OVERBROAD:
            cues = CONTEXT_CUES.get(m['class'], set())
            if any(cue in text.lower() for cue in cues):
                filtered.append(m)
        else:
            filtered.append(m)
    return filtered


# collect all mentions across the document


OUTPUT_JSON = Path("data/entity_mentions.json")

# --- Collect mentions ---
all_mentions = []

for chunk in raw_stream:  # raw_stream from your previous setup
    text = chunk['text']
    found = find_entities_in_text(text, alias_index)
    filtered = filter_overbroad_matches(found, text)
    
    for m in filtered:
        mention = {
            "docId": chunk['docId'],
            "page": chunk['page'],
            "section": chunk['section'],
            **m
        }
        all_mentions.append(mention)

# --- Save to JSON ---
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(all_mentions, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_mentions)} entity mentions to {OUTPUT_JSON}")

Saved 2689 entity mentions to data/entity_mentions.json


In [ ]:
# # --- Combined NER + KeyBERT Keyphrase Extraction → entities.jsonl -----------------
# import json
# from keybert import KeyBERT
# from pathlib import Path
# from collections import defaultdict

# # Paths
# ENTITY_MENTIONS_PATH = Path("data/entity_mentions.json")  # from your previous NER step
# RAW_TEXT_PATH = Path("data/raw_text.jsonl")
# OUTPUT_FINAL = Path("data/entities.jsonl")

# # --- 1. Load existing NER-based mentions ----------------------------------
# if ENTITY_MENTIONS_PATH.exists():
#     with ENTITY_MENTIONS_PATH.open(encoding="utf-8") as f:
#         all_mentions = json.load(f)
#     print(f"Loaded {len(all_mentions)} NER-based mentions.")
# else:
#     all_mentions = []
#     print("[WARN] entity_mentions.json not found. Proceeding with empty list.")

# # --- 2. Initialize KeyBERT model ------------------------------------------
# print("Initializing KeyBERT model (this may take a moment)...")
# kw_model = KeyBERT(model='all-MiniLM-L6-v2')  # lightweight & good semantic model

# # --- 3. Run KeyBERT on text chunks ----------------------------------------
# print("Running KeyBERT keyphrase extraction...")
# all_keyphrases = []
# raw_stream = iter_raw_text(RAW_TEXT_PATH, DEFAULT_DOC_ID)

# for chunk in raw_stream:
#     text = chunk["text"].strip()
#     if not text:
#         continue
#     try:
#         keywords = kw_model.extract_keywords(
#             text,
#             keyphrase_ngram_range=(1, 3),  # allow 1–3 word phrases
#             stop_words='english',
#             top_n=5                        # top 5 keyphrases per chunk
#         )
#     except Exception as e:
#         print(f"KeyBERT failed on page {chunk['page']}: {e}")
#         continue

#     for kw, score in keywords:
#         key_norm = norm(kw)
#         candidates = alias_index.get(key_norm)
#         if candidates:
#             # Phrase already exists in KB → linked
#             for ent in candidates:
#                 all_keyphrases.append({
#                     "docId": chunk["docId"],
#                     "page": chunk["page"],
#                     "section": chunk["section"],
#                     "text": kw,
#                     "score": float(score),
#                     "id": ent["id"],
#                     "class": ent["class"],
#                     "label": ent["label"],
#                     "source": "keyphrase_linked"
#                 })
#         else:
#             # 🚨 New candidate phrase (not found in KB)
#             all_keyphrases.append({
#                 "docId": chunk["docId"],
#                 "page": chunk["page"],
#                 "section": chunk["section"],
#                 "text": kw,
#                 "score": float(score),
#                 "id": None,
#                 "class": None,
#                 "label": None,
#                 "source": "keyphrase_new"
#             })

# print(f"Extracted {len(all_keyphrases)} keyphrase candidates (linked + new).")

# # --- 4. Merge NER mentions + keyphrase results ----------------------------
# combined = all_mentions + all_keyphrases
# print(f"Total combined records: {len(combined)}")

# # --- 5. Group by canonical entity ID --------------------------------------
# grouped = defaultdict(lambda: {
#     "id": None,
#     "label": None,
#     "class": None,
#     "aliases": [],
#     "mentions": [],
#     "new_keyphrases": []
# })

# for rec in combined:
#     cid = rec.get("id") or f"new:{norm(rec['text'])[:40]}"  # pseudo-ID for new phrases
#     group = grouped[cid]

#     # Fill metadata
#     if rec.get("id"):
#         group["id"] = rec["id"]
#         group["label"] = rec.get("label")
#         group["class"] = rec.get("class")
#     else:
#         group["class"] = "candidate"
#         group["label"] = rec["text"]

#     # Add mention or candidate phrase
#     if rec.get("source") == "keyphrase_new":
#         group["new_keyphrases"].append({
#             "docId": rec["docId"],
#             "page": rec["page"],
#             "section": rec["section"],
#             "text": rec["text"],
#             "score": rec["score"]
#         })
#     else:
#         group["mentions"].append({
#             "docId": rec["docId"],
#             "page": rec["page"],
#             "section": rec["section"],
#             "text": rec["text"],
#             "startChar": rec.get("start_char"),
#             "endChar": rec.get("end_char")
#         })

# # --- 6. Write out final entities.jsonl ------------------------------------
# with OUTPUT_FINAL.open("w", encoding="utf-8") as f:
#     for eid, data in grouped.items():
#         f.write(json.dumps(data, ensure_ascii=False) + "\n")

# print(f"\n✅ Wrote {len(grouped)} aggregated entities (NER + KeyBERT) to {OUTPUT_FINAL}")
